# Optimized Customer Support RAG — Xumo Stream Box
### (Advanced Chunking Strategies + Advanced Retriever Types)

This notebook builds on the basic Xumo Support RAG and **optimizes two specific stages**:

1. **Text Splitting** — smaller, more precise chunks tuned for FAQ-style support content
2. **Retriever** — a **hybrid (keyword + semantic) ensemble retriever**, wrapped with **contextual compression**, instead of plain similarity search

For every component in the pipeline, this notebook first explains **all the major types available in LangChain (Python)**, and then explains **why one specific type was chosen** — so the reasoning behind each decision is clear, not just the code.

At the end, there is a **side-by-side comparison** between the original (baseline) retriever setup and this optimized setup, run on the same questions, so you can see the practical difference.

**Stack (LangChain v1, current as of Sep 2026):**
- `langchain-community` — `PyPDFLoader`, `FAISS`, `BM25Retriever`
- `langchain-text-splitters` — `RecursiveCharacterTextSplitter`
- `langchain-huggingface` — open source embeddings
- `langchain-classic` — `EnsembleRetriever`, `ContextualCompressionRetriever` (advanced retrievers moved here in LangChain v1)
- `langchain-groq` via `init_chat_model` — LLM
- `langchain-core` — prompts, output parser, runnables, structured output

## 0. Setup — Install, Imports, PDF Path, LLM

In [ ]:
!pip install -q -U langchain langchain-core langchain-classic langchain-text-splitters langchain-community langchain-huggingface langchain-groq faiss-cpu sentence-transformers pypdf rank_bm25

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")  # community deprecation notice — harmless, package still works

from getpass import getpass

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain.chat_models import init_chat_model

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever

# Advanced retrievers live in langchain-classic in LangChain v1
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

from pydantic import BaseModel, Field
from typing import Literal

print("All imports successful")

In [ ]:
PDF_PATH = "xumo_stream_box_guide.pdf"

assert os.path.exists(PDF_PATH), f"PDF not found at {PDF_PATH} - update PDF_PATH above."
print("PDF found:", PDF_PATH)

In [ ]:
# Groq API key (free tier: https://console.groq.com/keys)
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

llm = init_chat_model("groq:llama-3.3-70b-versatile", temperature=0)
print(llm.invoke("Say hello in one short line").content)

## 1. Document Loader

### Types available in LangChain
| Loader | Best for |
|---|---|
| `TextLoader` | Plain `.txt` files |
| `PyPDFLoader` / `PyMuPDFLoader` | PDF files, one `Document` per page |
| `CSVLoader` | Tabular / spreadsheet data |
| `Docx2txtLoader` | Word documents |
| `WebBaseLoader` | Scraping a live web page |
| `DirectoryLoader` | Loading every file in a folder at once |
| `UnstructuredFileLoader` | Mixed/unknown file types (scanned docs, images, etc.) |

### Why `PyPDFLoader` is used here
The source document is a PDF, and `PyPDFLoader` loads it **one page at a time**, attaching page-number metadata (`page_label`) to every `Document`. This page metadata is valuable for a support bot — it lets the final answer reference exactly which page the information came from, which is useful for auditing or for a human agent to double-check.

In [ ]:
loader = PyPDFLoader(PDF_PATH)
pdf_docs = loader.load()

print("Total pages loaded:", len(pdf_docs))
print("\nPage 1 preview:\n", pdf_docs[0].page_content[:250])

## 2. Text Splitter — Optimization #1

### Types available in LangChain
| Splitter | How it works | Best for |
|---|---|---|
| `CharacterTextSplitter` | Splits on a single fixed separator (e.g. `\n\n`) | Simple, uniformly formatted text |
| `RecursiveCharacterTextSplitter` | Tries a list of separators in order (paragraph -> sentence -> word) until chunks fit | General-purpose default — preserves meaning better than a fixed splitter |
| `TokenTextSplitter` | Splits by LLM token count instead of characters | When you need chunks to precisely match a model's context window / token budget |
| `MarkdownHeaderTextSplitter` | Splits on `#`, `##`, `###` headers | Markdown docs, keeps each section together as one chunk |
| `HTMLHeaderTextSplitter` | Splits on HTML heading tags | Scraped web pages |
| `SemanticChunker` (`langchain_experimental`) | Uses embedding similarity to find natural topic breaks | Long, unstructured text where paragraph boundaries are unreliable |

### What changed from the baseline notebook, and why
The baseline notebook used `RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)`.

This document is **FAQ-style**: most answers are short, self-contained, and localized to 2-4 sentences (e.g. "Why is my activation code not working?"). Large 800-character chunks often merge **two or three unrelated Q&A entries into one chunk**, which adds noise and can confuse both the retriever and the LLM about which part of the chunk actually answers the question.

Here, the same `RecursiveCharacterTextSplitter` is kept (it is still the right general-purpose choice — it respects sentence and paragraph boundaries instead of cutting mid-word), but tuned smaller:
- `chunk_size=350` — small enough that each chunk usually maps to a single Q&A entry
- `chunk_overlap=50` — still enough overlap to avoid losing context at chunk boundaries

**Trade-off to be aware of:** smaller chunks improve retrieval *precision* (less irrelevant text per chunk) but can hurt *recall* if an answer genuinely needs more surrounding context than one small chunk provides. That is exactly why this notebook pairs smaller chunks with a stronger retriever (Section 5) instead of just shrinking the chunk size in isolation.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(pdf_docs)
print("Total chunks (optimized, chunk_size=350):", len(chunks))
print("(Baseline notebook produced far fewer, larger chunks at chunk_size=800)")
print("\nSample chunk:\n", chunks[5].page_content)

## 3. Embedding Model

### Types available in LangChain
| Embedding class | Provider | Notes |
|---|---|---|
| `OpenAIEmbeddings` | OpenAI (paid) | High quality, requires API key + cost per call |
| `CohereEmbeddings` | Cohere (paid) | Strong multilingual support |
| `GoogleGenerativeAIEmbeddings` | Google (paid) | Integrated with Gemini models |
| `HuggingFaceEmbeddings` | Open source (`sentence-transformers`) | Free, runs locally, no API key |
| `OllamaEmbeddings` | Open source, local via Ollama | Free, but requires Ollama running locally |

### Why `HuggingFaceEmbeddings` with `all-MiniLM-L6-v2` is used here
This is an **open source, locally-run** embedding model — no API key or per-call cost, which matters for a classroom/demo setting and for any support system processing a high volume of queries. `all-MiniLM-L6-v2` is small (~80MB) and fast, while still producing good-quality 384-dimensional embeddings for semantic similarity — a solid default for prototyping before considering a larger model for production.

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

test_vector = embedding_model.embed_query("How do I fix my remote?")
print("Embedding dimension:", len(test_vector))

## 4. Vector Store

### Types available in LangChain
| Vector store | Runs where | Notes |
|---|---|---|
| `FAISS` | Local, in-process | Free, no server, very fast for small/medium datasets |
| `Chroma` | Local (with persistence) or client-server | Popular for local RAG prototyping |
| `Pinecone` / `Weaviate` / `Qdrant` / `Milvus` | Managed cloud service | Built for large-scale, production, multi-user systems |
| `PGVector` | Inside a PostgreSQL database | Useful if the team already runs Postgres |
| `InMemoryVectorStore` | Local, in-process, not persisted | Quick tests only — data is lost when the process ends |

### Why `FAISS` is used here
For a single-document, single-machine support bot, `FAISS` needs no external server or account, saves its index to disk (`save_local` / `load_local`) so it does not need to be rebuilt on every run, and is fast enough for the small number of chunks this document produces. It is the same reason it was chosen in the baseline notebook — this component was not part of the optimization target.

In [ ]:
vectorstore = FAISS.from_documents(chunks, embedding_model)
vectorstore.save_local("xumo_faiss_index_optimized")

print("FAISS index built with", vectorstore.index.ntotal, "vectors")

## 5. Retriever — Optimization #2 (the main focus)

### Types available in LangChain
| Retriever type | How it works | Strength |
|---|---|---|
| **Similarity Search** (`search_type="similarity"`) | Returns the top-k chunks with the closest embedding vectors | Simple, fast baseline |
| **MMR** (`search_type="mmr"`) | Maximal Marginal Relevance — balances relevance *and* diversity so results aren't near-duplicates | Reduces redundant chunks in the context |
| **Similarity Score Threshold** (`search_type="similarity_score_threshold"`) | Only returns chunks above a minimum similarity score | Avoids forcing in irrelevant chunks when nothing matches well |
| **BM25Retriever** | Classic keyword/TF-IDF-style sparse search (no embeddings at all) | Excellent at exact keyword matches (product names, error codes, numbers) |
| **MultiQueryRetriever** | Uses an LLM to rewrite the user's question into several phrasings, then merges results | Improves recall for vaguely or unusually worded questions |
| **ContextualCompressionRetriever** | Wraps another retriever and uses an LLM to strip out irrelevant sentences from each retrieved chunk | Reduces noise and token usage sent to the final LLM call |
| **EnsembleRetriever** | Combines multiple retrievers (e.g. BM25 + vector) and merges/re-ranks their results | Hybrid search — captures both keyword matches and semantic meaning |
| **ParentDocumentRetriever** | Searches over small chunks, but returns the larger parent document/section they came from | Best of both worlds: precise search, full context in the answer |

### Why a hybrid Ensemble Retriever (BM25 + vector/MMR) is used here, wrapped in Contextual Compression

The baseline notebook used plain similarity search on the vector store alone. That has a real weakness for support documents: **customer queries often contain exact keywords** — "HDMI-1", "6-digit code", "Spectrum", "MAC address" — that a purely semantic embedding search can under-weight, because embeddings are optimized for *meaning*, not *exact terms*.

`BM25Retriever` is the opposite: it is excellent at exact keyword matches but has no understanding of meaning or paraphrasing (e.g. it would not connect "screen is black" to "standby mode" as well as a semantic model would).

Combining both with `EnsembleRetriever` gives the pipeline both strengths at once — this is the standard **hybrid search** pattern used in production RAG systems. The vector side is additionally set to `search_type="mmr"` so that, within the semantic results, near-duplicate chunks are pushed out in favor of more varied, complementary context.

Finally, the ensemble retriever is wrapped in a `ContextualCompressionRetriever` with an `LLMChainExtractor`. Because the retrievers above pull chunks whole, some retrieved chunks may only be partially relevant. The compressor uses the LLM to keep only the sentences that are actually relevant to the question before they are added to the prompt — this shrinks the context sent to the final LLM call, reduces cost, and reduces the chance of the model getting distracted by irrelevant text in an otherwise-relevant chunk.

In [ ]:
# 5a. Vector retriever with MMR (diversity-aware semantic search)
vector_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 15, "lambda_mult": 0.5},
)

# 5b. BM25 retriever (keyword/sparse search) - built directly from the same chunks
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4

# 5c. Ensemble retriever - hybrid search combining both
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],  # equal weight to keyword and semantic search
)

test_results = ensemble_retriever.invoke("activation code not working")
print("Ensemble retriever results:", len(test_results))
for i, doc in enumerate(test_results[:3]):
    print(f"--- Match {i+1} ---")
    print(doc.page_content[:150])
    print()

In [ ]:
# 5d. Wrap the ensemble retriever with contextual compression
compressor = LLMChainExtractor.from_llm(llm)

optimized_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever,
)

compressed_results = optimized_retriever.invoke("activation code not working")
print("Compressed results:", len(compressed_results))
for i, doc in enumerate(compressed_results):
    print(f"--- Compressed match {i+1} ---")
    print(doc.page_content)
    print()

## 6. Prompt Template

### Types available in LangChain
| Prompt type | Use case |
|---|---|
| `PromptTemplate` | Single plain-text prompt with `{variables}` |
| `ChatPromptTemplate` | A sequence of role-based messages (`system` / `human` / `ai`) — required for chat models |
| `FewShotPromptTemplate` | Includes worked examples in the prompt to guide the model's output format/style |
| `MessagesPlaceholder` | Injects a variable list of prior messages (e.g. conversation history) into a `ChatPromptTemplate` |

### Why `ChatPromptTemplate` is used here
The Groq LLM used in this notebook is a **chat model**, which expects a list of role-tagged messages rather than one flat string. A `system` message is used to fix the assistant's persona and ground-truth rule ("answer only from context"), and a separate `human` message carries the customer's question — keeping instructions and user input cleanly separated.

In [ ]:
support_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Xumo Stream Box customer support agent. "
     "Answer the customer's question using ONLY the context below. "
     "Be concise, friendly, and give clear step-by-step instructions when relevant. "
     "If the answer is not in the context, say you don't have that information and "
     "suggest contacting Xumo support via Start Chat - do not make anything up.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

## 7. Structured Output

### Types available in LangChain
| Approach | How it works |
|---|---|
| `with_structured_output(PydanticModel)` | Model is forced (via tool-calling or native structured output) to return data matching a Pydantic schema |
| `with_structured_output(TypedDict)` | Same, but returns a plain `dict` instead of a validated object |
| `with_structured_output(json_schema_dict)` | Same, using a raw JSON Schema instead of a Python class |

### Why a Pydantic model is used here
A Pydantic `BaseModel` gives both **field validation** (e.g. `category` can only be one of a fixed set of values via `Literal`) and a **typed Python object** back (`result.category`, not `result["category"]`), which is easier and safer to plug into a real support dashboard or CRM than an unvalidated string or raw dict.

In [ ]:
class SupportResponse(BaseModel):
    """Structured customer support response for a Xumo Stream Box query."""
    answer: str = Field(description="Clear, step-by-step answer to the customer's question")
    category: Literal[
        "activation", "wifi_connectivity", "remote_issue", "account_login",
        "hardware_issue", "software_update", "other"
    ] = Field(description="Best matching support category for this query")
    needs_human_escalation: bool = Field(
        description="True if the answer was not found in context and a human agent should step in"
    )

structured_llm = llm.with_structured_output(SupportResponse)

## 8. Output Parser

### Types available in LangChain
| Parser | Output type |
|---|---|
| `StrOutputParser` | Plain string (extracts `.content` from the model's `AIMessage`) |
| `JsonOutputParser` | Parses the model's text output into a Python `dict` |
| `PydanticOutputParser` | Parses text output into a validated Pydantic object (an older alternative to `with_structured_output`) |
| `CommaSeparatedListOutputParser` | Splits a comma-separated response into a Python list |

### Why `StrOutputParser` is used here
For the plain-text chat-widget version of the bot (as opposed to the structured CRM version in Section 7), the raw `AIMessage` object needs to become a plain string. `StrOutputParser` does exactly that with no extra parsing logic needed, since the LLM is not being asked to produce JSON in this branch.

In [ ]:
str_output_parser = StrOutputParser()

## 9. LLM

### Types available in LangChain (via `init_chat_model`)
| Provider string | Notes |
|---|---|
| `"openai:gpt-4o-mini"` | OpenAI, paid, very strong general quality |
| `"anthropic:claude-..."` | Anthropic, paid |
| `"groq:llama-3.3-70b-versatile"` | Groq, **free tier available**, extremely fast inference |
| `"ollama:llama3"` | Fully local, free, requires Ollama installed |

### Why Groq is used here
Groq offers a **free API tier** and very low-latency inference, which matters for a support-chat use case where response speed affects user experience, and matters for a classroom setting where students should not need a paid key to run this notebook.

In [ ]:
print(llm.invoke("In one sentence, what is hybrid search in RAG?").content)

## 10. The Optimized RAG Chain

Everything from Sections 2-9 combined: chunked with the smaller, FAQ-tuned splitter, retrieved with the hybrid ensemble + contextual compression retriever, then answered by the LLM.

In [ ]:
optimized_rag_chain = (
    RunnableParallel(
        context=optimized_retriever | RunnableLambda(format_docs),
        question=RunnablePassthrough(),
    )
    | support_prompt
    | llm
    | str_output_parser
)

def build_optimized_structured_chain():
    return (
        RunnableParallel(
            context=optimized_retriever | RunnableLambda(format_docs),
            question=RunnablePassthrough(),
        )
        | support_prompt
        | structured_llm
    )

optimized_structured_chain = build_optimized_structured_chain()

In [ ]:
test_questions = [
    "My activation code is not working, what should I do?",
    "The screen is black, how do I fix it?",
    "How do I connect my device to a new WiFi network?",
    "Do I need a Xumo account to use the Stream Box?",
]

for q in test_questions:
    print("Q:", q)
    print("A:", optimized_rag_chain.invoke(q))
    print("-" * 80)

## 11. Baseline vs Optimized — Side-by-Side Comparison

To make the effect of these two optimizations concrete, this section rebuilds the **original baseline setup** (large chunks, plain similarity search) next to the **optimized setup** from this notebook, and runs the same questions through both.

In [ ]:
# Baseline setup: large chunks + plain similarity search (same as the first notebook)
baseline_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
baseline_chunks = baseline_splitter.split_documents(pdf_docs)
baseline_vectorstore = FAISS.from_documents(baseline_chunks, embedding_model)
baseline_retriever = baseline_vectorstore.as_retriever(
    search_type="similarity", search_kwargs={"k": 4}
)

baseline_rag_chain = (
    RunnableParallel(
        context=baseline_retriever | RunnableLambda(format_docs),
        question=RunnablePassthrough(),
    )
    | support_prompt
    | llm
    | str_output_parser
)

print(f"Baseline chunks: {len(baseline_chunks)}   |   Optimized chunks: {len(chunks)}")

In [ ]:
comparison_question = "My activation code is not working, what should I do?"

print("=" * 30, "BASELINE (similarity search, chunk_size=800)", "=" * 30)
print(baseline_rag_chain.invoke(comparison_question))

print()
print("=" * 30, "OPTIMIZED (hybrid ensemble + compression, chunk_size=350)", "=" * 30)
print(optimized_rag_chain.invoke(comparison_question))

### Recap

| Stage | Baseline (previous notebook) | Optimized (this notebook) | Why the change helps |
|---|---|---|---|
| Chunking | `chunk_size=800`, generic | `chunk_size=350`, tuned for FAQ content | Each chunk maps closer to one self-contained Q&A, less noise per chunk |
| Retriever | Plain similarity search | `EnsembleRetriever` (BM25 + MMR vector) wrapped in `ContextualCompressionRetriever` | Hybrid search catches exact keywords *and* paraphrased meaning; compression strips irrelevant sentences before they reach the LLM |

Every other component (loader, embeddings, vector store, prompt, output parser, LLM) was kept the same and explained again here for completeness — the optimization target for this notebook was specifically chunking and retrieval, as requested.